# CodonFM (Encodon): tools tour + genetic-algorithm coding-sequence design

This notebook has two parts:

1. **Tools tour** — exercise all five CodonFM/Encodon proto-tools functions: `codonfm-fitness`, `codonfm-score`, `codonfm-embeddings`, `codonfm-gradient`, and `codonfm-sample`.
2. **Genetic-algorithm design** — optimize a coding sequence with **CodonFM as both the generator** (masked-codon resampling, a mutation generator) **and the constraint** (codon-level fitness / naturalness), using Proto Language's genetic-algorithm optimizer.

CodonFM is a *masked* (bidirectional) codon language model, so the natural optimizer is a population/mutation method (genetic algorithm or MCMC), not autoregressive beam search.

> **Requirements:** a CUDA GPU, and gated checkpoint access — set `HF_TOKEN` and accept the NVIDIA Open Model License on each `nvidia/NV-CodonFM-Encodon-*-v1` HuggingFace repo. Use `DEVICE = "cpu"` to run on CPU (slow).

In [ ]:
DEVICE = "cuda"
CHECKPOINT = "encodon_80m"

# A 24-codon in-frame coding-sequence fragment (GFP N-terminus) used as the seed throughout.
SEED_CDS = "ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCCTGGTCGAGCTGGACGGCGACGTAAAC"
assert len(SEED_CDS) % 3 == 0, "seed must be codon-aligned"
print(f"seed: {len(SEED_CDS)} nt / {len(SEED_CDS) // 3} codons")

## Part 1 — the five CodonFM tools

### `codonfm-fitness` — mean codon log-likelihood (naturalness)

In [ ]:
from proto_tools import CodonFMFitnessConfig, CodonFMFitnessInput, run_codonfm_fitness

fitness = run_codonfm_fitness(
    CodonFMFitnessInput(sequences=[SEED_CDS]),
    CodonFMFitnessConfig(model_checkpoint=CHECKPOINT, device=DEVICE),
)
print(f"fitness (mean codon log-likelihood): {fitness.results[0].fitness:.4f}")

### `codonfm-score` — ref-vs-alt codon log-likelihood ratio

Codon position 1 of the seed is `GTG` (Val). Compare a synonymous (`GTA`, Val) vs a missense (`AAA`, Lys) substitution.

In [ ]:
from proto_tools import CodonFMScoreConfig, CodonFMScoreInput, run_codonfm_score

scores = run_codonfm_score(
    CodonFMScoreInput(
        mutations=[
            {"sequence": SEED_CDS, "codon_position": 1, "ref_codon": "GTG", "alt_codon": "GTA"},
            {"sequence": SEED_CDS, "codon_position": 1, "ref_codon": "GTG", "alt_codon": "AAA"},
        ]
    ),
    CodonFMScoreConfig(model_checkpoint=CHECKPOINT, device=DEVICE),
)
for r in scores.results:
    print(f"{r.ref_codon}->{r.alt_codon}  llr={r.llr:+.4f}  (positive = reference favored)")

### `codonfm-embeddings` — CLS embedding

In [ ]:
from proto_tools import CodonFMEmbeddingsConfig, CodonFMEmbeddingsInput, run_codonfm_embeddings

emb = run_codonfm_embeddings(
    CodonFMEmbeddingsInput(sequences=[SEED_CDS]),
    CodonFMEmbeddingsConfig(model_checkpoint=CHECKPOINT, device=DEVICE),
)
print(f"CLS embedding dimension: {len(emb.results[0].embedding)}")

### `codonfm-gradient` — differentiable codon objective

Gradient of the masked codon NLL w.r.t. a relaxed `(L, 64)` codon distribution. `one_hot_codon_logits` builds a sharp start from the seed.

In [ ]:
from proto_tools import CodonFMGradientConfig, CodonFMGradientInput, run_codonfm_gradient
from proto_tools.tools.masked_models.codonfm import one_hot_codon_logits

grad = run_codonfm_gradient(
    CodonFMGradientInput(logits=one_hot_codon_logits(SEED_CDS, sharpness=2.0), temperature=0.6),
    CodonFMGradientConfig(model_checkpoint=CHECKPOINT, device=DEVICE),
)
print(f"loss (mean masked NLL): {grad.loss:.4f}")
print(f"gradient shape: {len(grad.gradient)} codons x {len(grad.gradient[0])} codon-vocab")

### `codonfm-sample` — masked-codon resampling (the mutation primitive)

In [ ]:
from proto_tools import CodonFMSampleConfig, CodonFMSampleInput, run_codonfm_sample

sampled = run_codonfm_sample(
    CodonFMSampleInput(sequences=[SEED_CDS]),
    CodonFMSampleConfig(model_checkpoint=CHECKPOINT, num_mutations=3, temperature=1.0, device=DEVICE, seed=0),
)
print(f"original: {SEED_CDS}")
print(f"resampled: {sampled.sequences[0]}  (length preserved: {len(sampled.sequences[0]) == len(SEED_CDS)})")

## Part 2 — genetic-algorithm design with CodonFM as generator + constraint

We optimize the coding sequence toward higher CodonFM naturalness. The **generator** (`CodonFMGenerator`) proposes mutations by masking and resampling codons; the **constraint** (`codonfm_fitness_constraint`, `direction="max"`) scores each candidate's fitness, mapping high fitness to low energy. The genetic-algorithm optimizer maintains a population, recombines parents, mutates offspring with the generator, and keeps the lowest-energy candidates.

In [ ]:
from proto_language.constraint import codonfm_fitness_constraint
from proto_language.core import Constraint, Construct, Program, Segment
from proto_language.generator import CodonFMGenerator, CodonFMGeneratorConfig
from proto_language.optimizer import GeneticAlgorithmOptimizer, GeneticAlgorithmOptimizerConfig

# One variable coding-sequence segment, seeded with the fragment above (a mutation generator
# refines an existing sequence, so the segment must start with a sequence).
segment = Segment(sequence=SEED_CDS, sequence_type="dna", label="cds")
construct = Construct([segment])

# Generator: CodonFM masked-codon resampling (resample 3 codons per mutation step).
generator = CodonFMGenerator(
    CodonFMGeneratorConfig(model_checkpoint=CHECKPOINT, num_mutations=3, temperature=1.0, device=DEVICE)
)
generator.assign(segment)

# Constraint: CodonFM fitness, maximize naturalness (high fitness -> low energy).
fitness_constraint = Constraint(
    inputs=[segment],
    function=codonfm_fitness_constraint,
    function_config={"model_checkpoint": CHECKPOINT, "direction": "max", "device": DEVICE},
)

In [ ]:
optimizer = GeneticAlgorithmOptimizer(
    constructs=[construct],
    generators=[generator],
    constraints=[fitness_constraint],
    config=GeneticAlgorithmOptimizerConfig(num_generations=8, population_size=16, num_results=4),
)

program = Program(optimizers=[optimizer], num_results=4)
program.run()

In [ ]:
# Compare the best designed sequence's fitness against the seed.
best = str(program.constructs[0].joined_sequences[0].sequence)
compare = run_codonfm_fitness(
    CodonFMFitnessInput(sequences=[SEED_CDS, best]),
    CodonFMFitnessConfig(model_checkpoint=CHECKPOINT, device=DEVICE),
)
seed_fit, best_fit = (r.fitness for r in compare.results)
print(f"seed fitness: {seed_fit:.4f}")
print(f"best fitness: {best_fit:.4f}  (delta {best_fit - seed_fit:+.4f})")
print(f"best sequence: {best}")